In [6]:
"""
Pet Knowledge RAG Data Collector
--------------------------------
Fetches and parses pet care articles (nutrition, health, behavior) from authoritative sources.
Outputs JSONL ready for vectorization and RAG pipeline.
"""

import requests
from bs4 import BeautifulSoup
import json
import time
from pathlib import Path
import random

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; PawtyBot/1.0; +https://pawty.ai)"}

# 1. 定义要抓取的网页列表
URLS = [
    # Nutrition
    ("https://www.merckvetmanual.com/management-and-nutrition/nutrition-small-animals/overview-of-nutrition-small-animals",
     "Merck Veterinary Manual", "nutrition", "cat/dog", "adult"),
    ("https://www.merckvetmanual.com/management-and-nutrition/nutrition-small-animals/dog-and-cat-foods",
     "Merck Veterinary Manual", "nutrition", "cat/dog", "all"),

    # General Care
    ("https://www.avma.org/resources-tools/pet-owners/petcare",
     "AVMA", "general care", "cat/dog", "all"),

    # Behavior / Mental Health
    ("https://www.merckvetmanual.com/behavior/behavior-of-dogs-and-cats/overview-of-behavior-of-dogs-and-cats",
     "Merck Veterinary Manual", "behavior", "cat/dog", "all")
]

def fetch_html(url):
    """请求网页并返回Soup对象"""
    for attempt in range(3):
        try:
            r = requests.get(url, headers=HEADERS, timeout=15)
            if r.status_code == 200:
                return BeautifulSoup(r.text, "html.parser")
            else:
                print(f"[WARN] {url} returned {r.status_code}")
        except Exception as e:
            print(f"[ERROR] Attempt {attempt+1}: {e}")
        time.sleep(2 + random.random()*2)
    return None

def clean_text(t):
    return ' '.join(t.split())

def extract_paragraphs(soup):
    """提取主要段落内容"""
    paras = []
    for tag in soup.find_all(["p", "li"]):
        text = clean_text(tag.get_text())
        if len(text) > 80 and "cookie" not in text.lower():
            paras.append(text)
    return paras



def save_jsonl(records, path="data/pet_rag_articles.jsonl"):
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)  # ensure ./data exists

    with output_path.open("a", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def main():
    all_records = []
    for url, source, topic, species, life_stage in URLS:
        print(f"Fetching: {url}")
        soup = fetch_html(url)
        if not soup:
            continue
        title_tag = soup.find("h1") or soup.find("title")
        title = clean_text(title_tag.get_text()) if title_tag else "Untitled"
        paragraphs = extract_paragraphs(soup)
        if not paragraphs:
            print(f"[WARN] No text found for {url}")
            continue

        # 简短摘要
        summary = " ".join(paragraphs[:3])[:1000]

        record = {
            "title": title,
            "summary": summary,
            "content": "\n".join(paragraphs),
            "topic": topic,
            "species": species,
            "life_stage": life_stage,
            "source_name": source,
            "source_url": url,
            "crawl_date": time.strftime("%Y-%m-%d"),
        }
        all_records.append(record)

        print(f"[OK] {title} ({len(paragraphs)} paragraphs)")
        time.sleep(2 + random.random()*3)

    save_jsonl(all_records)
    print(f"✅ Saved {len(all_records)} records to data/pet_rag_articles.jsonl")

if __name__ == "__main__":
    main()


Fetching: https://www.merckvetmanual.com/management-and-nutrition/nutrition-small-animals/overview-of-nutrition-small-animals
[OK] Overview of Nutrition: Small Animals (17 paragraphs)
Fetching: https://www.merckvetmanual.com/management-and-nutrition/nutrition-small-animals/dog-and-cat-foods
[OK] Dog and Cat Foods (67 paragraphs)
Fetching: https://www.avma.org/resources-tools/pet-owners/petcare
[OK] Pet care (50 paragraphs)
Fetching: https://www.merckvetmanual.com/behavior/behavior-of-dogs-and-cats/overview-of-behavior-of-dogs-and-cats
[WARN] https://www.merckvetmanual.com/behavior/behavior-of-dogs-and-cats/overview-of-behavior-of-dogs-and-cats returned 404
[WARN] https://www.merckvetmanual.com/behavior/behavior-of-dogs-and-cats/overview-of-behavior-of-dogs-and-cats returned 404
[WARN] https://www.merckvetmanual.com/behavior/behavior-of-dogs-and-cats/overview-of-behavior-of-dogs-and-cats returned 404
✅ Saved 3 records to data/pet_rag_articles.jsonl
